# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhijeetpayal16-del/FLY-RANK-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*The unit of analysis is a single (client_id, search_target_id, date) tuple, meaning one row represents the aggregated search performance metrics for a specific target on a single day. The core time window for analysis spans from 2026-01-01 through 2026-05-31 (a mid-panel history window), excluding the final sealed month of June 2026 which serves as our out-of-time test frame.

In [1]:
# Section 1: Verify the row grain count and distinct time spans
try:
    grain_check = con.execute("""
        SELECT
            MIN(date) as min_date,
            MAX(date) as max_date,
            COUNT(*) as total_rows,
            COUNT(DISTINCT (client_id || '-' || search_target_id || '-' || date)) as unique_grain_rows
        FROM fact_daily_sample
    """).df()
    print("Unit of analysis snapshot:")
    display(grain_check)
except Exception as e:
    print("Database connection check — make sure the warehouse initialization ran at the top of the notebook:", e)

Database connection check — make sure the warehouse initialization ran at the top of the notebook: name 'con' is not defined


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*:

Features: impressions, position, session_depth (historical rollups capturing past target prominence).

Label: clicks (used to construct the proxy score target for high-value user conversion frames).

Context: client_id, search_target_id, date (identifying constraints used exclusively to anchor the grain).

Excluded: raw_query_strings or heavy unstructured tracking telemetry. Why: They introduce high cardinality tracking overhead and risk systemic data leakage if unique strings are evaluated directly at the decision moment.

In [2]:
# Section 2: Schema verification check
try:
    schema_cols = con.execute("DESCRIBE SELECT * FROM fact_daily_sample LIMIT 1").df()
    print("Verified Warehouse Columns:")
    display(schema_cols[['column_name', 'column_type']])
except Exception as e:
    print("Database connection check:", e)

Database connection check: name 'con' is not defined


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
# Section 3: The Three Contract Verification Facts
try:
    print("--- Fact 1: Grain Uniqueness Verification ---")
    grain_dupes = con.execute("""
        SELECT client_id, search_target_id, date, COUNT(*) as occurrence_count
        FROM fact_daily_sample
        GROUP BY 1, 2, 3
        HAVING occurrence_count > 1
        LIMIT 5
    """).df()
    print(f"Duplicate rows found at this grain: {len(grain_dupes)}")

    print("\n--- Fact 2: Volume & Date Span Verification ---")
    volume_stats = con.execute("""
        SELECT COUNT(*) as total_records, COUNT(DISTINCT date) as unique_days
        FROM fact_daily_sample
    """).df()
    display(volume_stats)

    print("\n--- Fact 3: Availability Filter (IS TRUE Verification) ---")
    availability_check = con.execute("""
        SELECT COUNT(*) as active_high_engagement_rows
        FROM fact_daily_sample
        WHERE (clicks > 0) IS TRUE
    """).df()
    display(availability_check)
except Exception as e:
    print("Ensure database session 'con' is initialized at the top:", e)

--- Fact 1: Grain Uniqueness Verification ---
Ensure database session 'con' is initialized at the top: name 'con' is not defined


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*The data has clear boundary limits. First, it reflects an unbalanced snapshot of historical interactions that cannot capture unseen target configurations or completely new user intents. Second, because it focuses on warehouse aggregate metrics, it lacks raw session telemetry like real-time hovering patterns or detailed bounce dynamics. Lastly, relying on a fixed monthly window means it cannot dynamically account for long-term seasonal trends or sudden external market shifts.

In [4]:
# Section 4: Quantifying the boundaries of the data slice
try:
    # Let's count null values across core fields to inspect data limits
    limits_check = con.execute("""
        SELECT
            COUNT(*) - COUNT(client_id) as missing_clients,
            COUNT(*) - COUNT(search_target_id) as missing_targets,
            COUNT(*) - COUNT(clicks) as missing_clicks
        FROM fact_daily_sample
    """).df()
    print("Data Integrity & Completeness Profile:")
    display(limits_check)
except Exception as e:
    print("Database session 'con' not active. Ensure your top-level initialization cell ran successfully.")
    print("Error summary:", e)

Database session 'con' not active. Ensure your top-level initialization cell ran successfully.
Error summary: name 'con' is not defined


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.